In [6]:
# -*- coding: utf-8 -*-

import os
import numpy as np
import time
import argparse
import copy
import pandas as pd
from vgg import *
from math import ceil
from random import Random

import torch
import torch.distributed as dist
import torch.utils.data.distributed
import torch.nn as nn
import torch.nn.functional as F
from torch.multiprocessing import Process
import torchvision
from torchvision import datasets, transforms
import torch.backends.cudnn as cudnn
import torchvision.models as models

from FedNova import *
#import util_v4 as util


In [12]:
def args_parser():
    parser = argparse.ArgumentParser()
    parser.add_argument('--exp_name','-n', 
                        default="test_fednova", 
                        type=str, 
                        help='experiment name, used for saving results')
    parser.add_argument('--backend',
                        default="nccl",
                        type=str,
                        help='background name')
    parser.add_argument('--model', 
                        default="VGG", 
                        type=str, 
                        help='neural network model')
    parser.add_argument('--NIID',
                        action='store_true',
                        default = True,
                        help='whether the dataset is non-iid or not')
    parser.add_argument('--alpha', 
                        default=0.5,  # default 0.2
                        type=float, 
                        help='control the non-iidness of dataset')
    parser.add_argument('--gmf', 
                        default=0, 
                        type=float, 
                        help='global (server) momentum factor')
    parser.add_argument('--lr', 
                        default=0.01, 
                        type=float, 
                        help='client learning rate')
    parser.add_argument('--momentum', 
                        default=0.0, 
                        type=float, 
                        help='local (client) momentum factor')
    parser.add_argument('--bs', 
                        default=16, # default 32
                        type=int, 
                        help='batch size on each worker/client')
    parser.add_argument('--rounds', 
                        default=5, # default 200
                        type=int, 
                        help='total coommunication rounds')
    parser.add_argument('--localE', 
                        default=10, 
                        type=int, 
                        help='number of local epochs')
    parser.add_argument('--meanE', 
                        default=5,
                        type=int, 
                        help='average number of local epochs')
    parser.add_argument('--print_freq', 
                        default=100, 
                        type=int, 
                        help='print info frequency')
    parser.add_argument('--size', 
                        default=2,  # default 16
                        type=int, 
                        help='number of local workers')
# =============================================================================
#     parser.add_argument('--rank', 
#                         default=0, 
#                         type=int, 
#                         help='the rank of worker')
# =============================================================================
    parser.add_argument('--seed', 
                        default=1, 
                        type=int, 
                        help='random seed')
    parser.add_argument('--save', '-s', 
                        action='store_true', 
                        help='whether save the training results')
    parser.add_argument('--p', '-p', 
                        action='store_true', 
                        help='whether the dataset is partitioned or not')
    parser.add_argument('--pattern',
                        type=str, 
                        help='pattern of local steps')
    parser.add_argument('--optimizer', 
                        default='fednova', 
                        type=str, 
                        help='optimizer name')
    parser.add_argument('--initmethod',
                        default='tcp://h0:22000',
                        type=str,
                        help='init method')
    parser.add_argument('--mu', 
                        default=1, 
                        type=float, 
                        help='mu parameter in fedprox')
    parser.add_argument('--savepath',
                        default='./results/',
                        type=str,
                        help='directory to save exp results')
    parser.add_argument('--datapath',
                        default='./data/',
                        type=str,
                        help='directory to load data')
    args = parser.parse_known_args()[0]
    return args

def update_learning_rate(optimizer, epoch, target_lr):
    """
    1) Decay learning rate exponentially (epochs 30, 60, 80)
    ** note: target_lr is the reference learning rate from which to scale down
    """
    if epoch == int(args.rounds / 2):
        lr = target_lr/10
        print('Updating learning rate to {}'.format(lr))
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

    if epoch == int(args.rounds * 0.75):
        lr = target_lr/100
        print('Updating learning rate to {}'.format(lr))
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

class Meter(object):
    """ Computes and stores the average, variance, and current value """

    def __init__(self, init_dict=None, ptag='Time', stateful=False,
                 csv_format=True):
        """
        :param init_dict: Dictionary to initialize meter values
        :param ptag: Print tag used in __str__() to identify meter
        :param stateful: Whether to store value history and compute MAD
        """
        self.reset()
        self.ptag = ptag
        self.value_history = None
        self.stateful = stateful
        if self.stateful:
            self.value_history = []
        self.csv_format = csv_format
        if init_dict is not None:
            for key in init_dict:
                try:
                    # TODO: add type checking to init_dict values
                    self.__dict__[key] = init_dict[key]
                except Exception:
                    print('(Warning) Invalid key {} in init_dict'.format(key))

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        self.std = 0
        self.sqsum = 0
        self.mad = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count
        self.sqsum += (val ** 2) * n
        if self.count > 1:
            self.std = ((self.sqsum - (self.sum ** 2) / self.count)
                        / (self.count - 1)
                        ) ** 0.5
        if self.stateful:
            self.value_history.append(val)
            mad = 0
            for v in self.value_history:
                mad += abs(v - self.avg)
            self.mad = mad / len(self.value_history)

    def __str__(self):
        if self.csv_format:
            if self.stateful:
                return str('{dm.val:.3f},{dm.avg:.3f},{dm.mad:.3f}'
                           .format(dm=self))
            else:
                return str('{dm.val:.3f},{dm.avg:.3f},{dm.std:.3f}'
                           .format(dm=self))
        else:
            if self.stateful:
                return str(self.ptag) + \
                       str(': {dm.val:.3f} ({dm.avg:.3f} +- {dm.mad:.3f})'
                           .format(dm=self))
            else:
                return str(self.ptag) + \
                       str(': {dm.val:.3f} ({dm.avg:.3f} +- {dm.std:.3f})'
                           .format(dm=self))
   

class Partition(object):
    """ Dataset-like object, but only access a subset of it. """

    def __init__(self, data, index):
        self.data = data
        self.index = index

    def __len__(self):
        return len(self.index)

    def __getitem__(self, index):
        data_idx = self.index[index]
        return self.data[data_idx]

class DataPartitioner(object):
    """ Partitions a dataset into different chuncks. """
    def __init__(self, data, sizes=[0.7, 0.2, 0.1], seed=1234, isNonIID=False, alpha=0, dataset=None):
        self.data = data
        self.dataset = dataset
        if isNonIID:
            print('Dataset is Non IID!')
            self.partitions, self.ratio = self.__getDirichletData__(data, sizes, seed, alpha)

        else:
            print('Dataset is IID!')
            self.partitions = [] 
            self.ratio = sizes
            rng = Random() 
            rng.seed(seed) 
            data_len = len(data) 
            indexes = [x for x in range(0, data_len)] 
            rng.shuffle(indexes) 
             
     
            for frac in sizes: 
                part_len = int(frac * data_len)
                self.partitions.append(indexes[0:part_len])
                indexes = indexes[part_len:]

        

    def use(self, partition):
        return Partition(self.data, self.partitions[partition])

    def __getNonIIDdata__(self, data, sizes, seed, alpha):
        labelList = data.train_labels
        rng = Random()
        rng.seed(seed)
        a = [(label, idx) for idx, label in enumerate(labelList)]
        # Same Part
        labelIdxDict = dict()
        for label, idx in a:
            labelIdxDict.setdefault(label,[])
            labelIdxDict[label].append(idx)
        labelNum = len(labelIdxDict)
        labelNameList = [key for key in labelIdxDict]
        labelIdxPointer = [0] * labelNum
        # sizes = number of nodes
        partitions = [list() for i in range(len(sizes))]
        eachPartitionLen= int(len(labelList)/len(sizes))
        # majorLabelNumPerPartition = ceil(labelNum/len(partitions))
        majorLabelNumPerPartition = 2
        basicLabelRatio = alpha

        interval = 1
        labelPointer = 0

        # basic part
        for partPointer in range(len(partitions)):
            requiredLabelList = list()
            for _ in range(majorLabelNumPerPartition):
                requiredLabelList.append(labelPointer)
                labelPointer += interval
                if labelPointer > labelNum - 1:
                    labelPointer = interval
                    interval += 1
            for labelIdx in requiredLabelList:
                start = labelIdxPointer[labelIdx]
                idxIncrement = int(basicLabelRatio*len(labelIdxDict[labelNameList[labelIdx]]))
                partitions[partPointer].extend(labelIdxDict[labelNameList[labelIdx]][start:start+ idxIncrement])
                labelIdxPointer[labelIdx] += idxIncrement

        #random part
        remainLabels = list()
        for labelIdx in range(labelNum):
            remainLabels.extend(labelIdxDict[labelNameList[labelIdx]][labelIdxPointer[labelIdx]:])
        rng.shuffle(remainLabels)
        for partPointer in range(len(partitions)):
            idxIncrement = eachPartitionLen - len(partitions[partPointer])
            partitions[partPointer].extend(remainLabels[:idxIncrement])
            rng.shuffle(partitions[partPointer])
            remainLabels = remainLabels[idxIncrement:]

        return partitions
    
    def __getDirichletData__(self, data, psizes, seed, alpha):
        n_nets = len(psizes)
        K = 10
        # labelList = np.array(data.train_labels) 修改成下面一行
        labelList = np.array(data.targets)
        min_size = 0
        N = len(labelList)
        np.random.seed(2020)

        net_dataidx_map = {}
        while min_size < K:
            idx_batch = [[] for _ in range(n_nets)]
            # for each class in the dataset
            for k in range(K):
                idx_k = np.where(labelList == k)[0]
                np.random.shuffle(idx_k)
                proportions = np.random.dirichlet(np.repeat(alpha, n_nets)) # alpha 越大越接近iid
                ## Balance
                proportions = np.array([p*(len(idx_j)<N/n_nets) for p,idx_j in zip(proportions,idx_batch)])
                proportions = proportions/proportions.sum()
                proportions = (np.cumsum(proportions)*len(idx_k)).astype(int)[:-1]
                idx_batch = [idx_j + idx.tolist() for idx_j,idx in zip(idx_batch,np.split(idx_k,proportions))]
                min_size = min([len(idx_j) for idx_j in idx_batch])

        for j in range(n_nets):
            np.random.shuffle(idx_batch[j])
            net_dataidx_map[j] = idx_batch[j]
            
        net_cls_counts = {}

        for net_i, dataidx in net_dataidx_map.items():
            unq, unq_cnt = np.unique(labelList[dataidx], return_counts=True)
            tmp = {unq[i]: unq_cnt[i] for i in range(len(unq))}
            net_cls_counts[net_i] = tmp
        # print('Data statistics: %s' % str(net_cls_counts))

        local_sizes = []
        for i in range(n_nets):
            local_sizes.append(len(net_dataidx_map[i]))
        local_sizes = np.array(local_sizes)
        weights = local_sizes/np.sum(local_sizes)
        # print(weights)

        return idx_batch, weights

def partition_dataset(rank, size, args):
    print('==> load train data')
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    trainset = torchvision.datasets.CIFAR10(root=args.datapath, 
                                            train=True, 
                                            download=False, 
                                            transform=transform_train)
    
    partition_sizes = [1.0 / size for _ in range(size)]
    partition = DataPartitioner(trainset, partition_sizes, isNonIID=args.NIID, alpha=args.alpha)
    ratio = partition.ratio
    partition = partition.use(rank)
    train_loader = torch.utils.data.DataLoader(partition, 
                                            batch_size=args.bs, 
                                            shuffle=True, 
                                            num_workers=size,
                                            pin_memory=True)

    print('==> load test data')
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    testset = torchvision.datasets.CIFAR10(root=args.datapath, 
                                        train=False, 
                                        download=False, 
                                        transform=transform_test)
    test_loader = torch.utils.data.DataLoader(testset, 
                                            batch_size=64, 
                                            shuffle=False, 
                                            num_workers=size)

    # You can add more datasets here
    return train_loader, test_loader, ratio

def select_model(num_class, args):
    if args.model == 'VGG':
        model = vgg11()

    # You can add more models here
    return model

def comp_accuracy(output, target, topk=(1,)):
    """Computes the accuracy over the k top predictions for the specified values of k"""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].view(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res 

def train(model, criterion, optimizer, loader, epoch):

    model.train()

    losses = Meter(ptag='Loss')
    top1 = Meter(ptag='Prec@1')
    for batch_idx, (data, target) in enumerate(loader):
        # data loading
        data = data.cuda(non_blocking = True)
        target = target.cuda(non_blocking = True)

        # forward pass
        output = model(data)
        loss = criterion(output, target)

        # backward pass
        loss.backward()
        
        # total_norm = 0.0
        # for param in model.parameters():
        #     if param.grad is not None:
        #         param_norm = param.grad.data.norm(2)  # L2范数
        #         total_norm += param_norm.item() ** 2
        # grad_norm = total_norm ** 0.5
        # grad_norms.append(grad_norm)  # 记录梯度范数

        # gradient step
        optimizer.step()
        optimizer.zero_grad()

        # write log files
        train_acc = comp_accuracy(output, target)
        

        losses.update(loss.item(), data.size(0))
        top1.update(train_acc[0].item(), data.size(0))
    
    return model.state_dict()

def average_weights(w):
    """
    Returns the average of the weights.
    """
    w_avg = copy.deepcopy(w[0])
    for key in w_avg.keys():
        for i in range(1, len(w)):
            w_avg[key] += w[i][key]
        w_avg[key] = torch.div(w_avg[key], len(w))
    return w_avg       

def evaluate(model, test_loader):
    model.eval()
    top1 = Meter(ptag='Acc')
    loss=0
    with torch.no_grad():
        for data, target in test_loader:
            data = data.cuda(non_blocking = True)
            target = target.cuda(non_blocking = True)
            outputs = model(data)
            acc1 = comp_accuracy(outputs, target)
            top1.update(acc1[0].item(), data.size(0))
            batch_loss = criterion(outputs, target)
            loss += batch_loss.item()

    return top1.avg, loss


def generate_random_array(size, mean, lower_bound, upper_bound):
    # Generate a random array within the specified range
    random_array = np.random.randint(lower_bound, upper_bound + 1, size)
    
    # Calculate the current mean of the array
    current_mean = np.mean(random_array)
    
    # Adjust the array to have the desired mean
    adjustment_factor = mean / current_mean
    adjusted_array = np.clip(random_array * adjustment_factor, lower_bound, upper_bound).astype(int)
    
    # Ensure that the mean is as close to the target as possible
    while abs(np.mean(adjusted_array) - mean) > 0.1:  # Adjust the threshold as needed
        random_array = np.random.randint(lower_bound, upper_bound + 1, size)
        current_mean = np.mean(random_array)
        adjustment_factor = mean / current_mean
        adjusted_array = np.clip(random_array * adjustment_factor, lower_bound, upper_bound).astype(int)

    return adjusted_array

# gpt 版本
def get_Fnw(global_model, train_loader):
    global_model.train()  # 确保模型在评估模式（关闭Dropout/BatchNorm）
    total_grad = None
    total_samples = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        images.requires_grad_(True)
        # 前向传播并计算梯度
        global_model.zero_grad()
        target_pred = global_model(images)
        loss = criterion(target_pred, labels)
        gradients = torch.autograd.grad(loss, global_model.parameters(), create_graph=False)

        # 扁平化并累加梯度
        batch_grad = torch.cat([g.flatten() for g in gradients])
        if total_grad is None:
            total_grad = batch_grad
        else:
            total_grad += batch_grad
        total_samples += 1

    # 计算平均梯度
    mean_grad = total_grad / total_samples
    return mean_grad.unsqueeze(0)  # 保持与原代码输出形状一致

def get_betan(rank):
    Fnw=means[rank] #mean_rank = get_Fnw(global_model, train_loader), means.append(mean_rank)
    # x = Fnw-Fw
    # x_norm = x.norm(dim=1, p=1)
    # Fw_norm = Fw.norm(dim=1, p=1)
    # beta = x_norm/Fw_norm
    
    Fnw_norm = Fnw.norm(dim=1, p=2)
    Fw_norm = Fw.norm(dim=1, p=2)
    beta = Fnw_norm/Fw_norm -1
    
    beta = beta.detach().cpu().numpy()
    return beta[0]

In [13]:
# # yn new:
# def get_Fnw(global_model, train_loader):
#     gradients = []
#     for batch_idx, (images, labels) in enumerate(train_loader):
#         images, labels = images.to(device), labels.to(device)
#         #print(images.size(), labels.size())
#         images.requires_grad = True

#         global_model.zero_grad()
#         target_pred = global_model(images)
#         y = criterion(target_pred, labels)
#         dy = torch.autograd.grad(y, global_model.parameters())
#         target_grad = torch.cat([g.detach().clone().reshape(1, -1) for g in dy], 1).reshape(1, -1)
#         target_grad.requires_grad = True
#         target_grad = target_grad.to(device)
#         #grad_norm = torch.norm(target_grad).detach().cpu().numpy()

#         gradients.append(target_grad)
#     mean = torch.mean(torch.stack(gradients), dim=0)
#     #print(mean.size())
#     return mean

# def get_betan(rank):
#     Fnw=means[rank] #mean_rank = get_Fnw(global_model, train_loader), means.append(mean_rank)
#     Fnw_norm = Fnw.norm(dim=1, p=2)
#     Fw_norm = Fw.norm(dim=1, p=2)
    
#     beta = Fnw_norm/Fw_norm
#     beta = beta.detach().cpu().numpy()
#     return beta[0]

# split main fuc

In [15]:
args = args_parser()
save_pth = False
if not os.path.exists('./log/'+args.exp_name):
    os.makedirs('./log/'+args.exp_name) 
#rank = args.rank#the rank of worker,0,because distributed runing, each file has a client id
size = args.size #number of local workers,8
device = 'cuda:0'

torch.manual_seed(args.seed)
torch.cuda.manual_seed(args.seed)
torch.backends.cudnn.deterministic = True

global_model = select_model(10, args).to(device)#10 class classification
global_weights = global_model.state_dict()
algorithms = {
    # 'fedavg': FedProx, # mu = 0
    # 'fedprox': FedProx,
    # 'scaffold': Scaffold,
    'fednova': FedNova,
    # 'fednova_vr':FedNovaVR,
}
selected_opt = algorithms[args.optimizer]
epochs_array = generate_random_array(size, mean=args.meanE, lower_bound=1, upper_bound=args.localE)
# epochs_array = np.array([args.meanE]*args.size)
print(epochs_array)
# test_accs,test_losses=[],[]
test_accs,test_losses, test_Fw_norm=[],[],[]
betas = np.zeros([args.rounds,args.size]) 
results = pd.DataFrame()
for rnd in range(args.rounds): # 轮次 200 rounds
    start = time.time()
    global_model.train()
    local_weights=[]
    means =[]
    for rank in range(args.size): # 遍历所有客户端
        train_loader, test_loader, DataRatios = partition_dataset(rank, size, args)
        local_epochs = epochs_array[rank]
        print('global round:{:d}, client id={:d}, localE={:d}'.format(rnd, rank, local_epochs))

        local_model = copy.deepcopy(global_model)
        criterion = nn.CrossEntropyLoss().to(device)
        optimizer = selected_opt(local_model.parameters(),
                         lr=args.lr,
                         gmf=args.gmf,
                         mu=args.mu,
                         ratio=DataRatios[rank],
                         momentum=args.momentum,
                         nesterov = False,
                         weight_decay=1e-4)

        update_learning_rate(optimizer, rnd, args.lr)


        for t in range(local_epochs): # epoch 96
            w =train(local_model, criterion, optimizer, train_loader, t)
            local_weights.append(copy.deepcopy(w))

        mean_rank = get_Fnw(global_model, train_loader) 
        means.append(mean_rank)
    Fw = torch.mean(torch.stack(means), dim=0) #每轮一个
    Fw_norm = Fw.norm(dim=1, p=2)
    test_Fw_norm.append(Fw_norm)
    Fw_norm_values = [tensor.item() for tensor in test_Fw_norm]
    for rank in range(args.size):
        beta = get_betan(rank)  # 使用当前轮的 Fw 和 means[rank]
        betas[rnd, rank] = beta #
    global_weights = average_weights(local_weights)
    global_model.load_state_dict(global_weights)
    test_acc, test_loss = evaluate(global_model, test_loader)
    print('global round:{:d}, test_acc={:.2f}, test_loss={:.4f}'.format(rnd, test_acc, test_loss))
    test_accs.append(test_acc)
    test_losses.append(test_loss)
    end = time.time()
    cost_time = end -start
    print(f'Round {rnd} takes {cost_time} s.')
    # save model
    file_name = f'./result/exp_{args.optimizer}_{args.NIID}_alpha{args.alpha}_bs{args.bs}_epoch{args.meanE}_{args.localE}_size{args.size}'
    if save_pth:
        model_path = f'{file_name}_model.pth'
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        if os.path.exists(model_path):
            print(f"Warning: Model file already exists at {model_path}. Skipping save.")
        else:
            torch.save(global_model.state_dict(), model_path)
            print(f"Round {rnd} model saved to {model_path}")
# save results
results['round'] = list(range(len(test_accs)))
results['acc'] =test_accs
results['loss'] =test_losses
results['Fw_norm'] =Fw_norm_values

file_name = f'./result/exp_{args.optimizer}_alpha{args.alpha}_lr{args.lr}_bs{args.bs}_rounds{len(test_accs)}_epoch{args.meanE}_{args.localE}_size{args.size}.csv'
os.makedirs(os.path.dirname(file_name), exist_ok=True)
results.to_csv(file_name, index=False)
pd.DataFrame(betas).to_csv(f'{file_name}_beta.csv', index=False)



[2 8]
==> load train data
Dataset is Non IID!
==> load test data
global round:0, client id=0, localE=2
==> load train data
Dataset is Non IID!
==> load test data
global round:0, client id=1, localE=8
global round:0, test_acc=10.00, test_loss=370.0794
Round 0 takes 76.5265634059906 s.
==> load train data
Dataset is Non IID!
==> load test data
global round:1, client id=0, localE=2
==> load train data
Dataset is Non IID!
==> load test data
global round:1, client id=1, localE=8
global round:1, test_acc=14.00, test_loss=383.4402
Round 1 takes 78.02366900444031 s.
==> load train data
Dataset is Non IID!
==> load test data
global round:2, client id=0, localE=2
Updating learning rate to 0.001
==> load train data
Dataset is Non IID!
==> load test data
global round:2, client id=1, localE=8
Updating learning rate to 0.001
global round:2, test_acc=10.00, test_loss=395.7430
Round 2 takes 77.98168516159058 s.
==> load train data
Dataset is Non IID!
==> load test data
global round:3, client id=0, loc

In [ ]:
results['round'] = list(range(len(test_accs)))
results['acc'] =test_accs
results['loss'] =test_losses
results['Fw_norm'] =Fw_norm_values
file_name = f'./result/{args.optimizer}_alpha{args.alpha}_lr{args.lr}_bs{args.bs}_rounds{len(test_accs)}_epoch{args.meanE}_{args.localE}_size{args.size}.csv'
results.to_csv(file_name, index = False)

pd.DataFrame(betas).to_csv(f'{file_name}_beta.csv', index=False)



# raw main func

In [ ]:
# if __name__ == "__main__":
    
    
#     args = args_parser()
#     if not os.path.exists('./log/'+args.exp_name):
#         os.makedirs('./log/'+args.exp_name) # yn modify from mkdir to makedirs
#     #rank = args.rank#the rank of worker,0,because distributed runing, each file has a client id
#     size = args.size #number of local workers,8
#     device = 'cuda:0'
    
    
#     torch.manual_seed(args.seed)
#     torch.cuda.manual_seed(args.seed)
#     torch.backends.cudnn.deterministic = True
    
#     global_model = select_model(10, args).to(device)#10 class classification
#     global_weights = global_model.state_dict()
    
    
#     algorithms = {
#         'fedavg': FedProx, # mu = 0
#         'fedprox': FedProx,
#         # 'scaffold': Scaffold,
#         'fednova': FedNova,
#         # 'fednova_vr':FedNovaVR,
#     }
    
#     selected_opt = algorithms[args.optimizer]
    
#     epochs_array = generate_random_array(size, mean=args.meanE, lower_bound=1, upper_bound=args.localE)
    
#     test_accs,test_losses=[],[]
#     for rnd in range(args.rounds):
#         global_model.train()
#         local_weights=[]
#         gradient_norm =[]
        
#         for rank in range(args.size):
#             train_loader, test_loader, DataRatios = partition_dataset(rank, size, args)
#             local_epochs = epochs_array[rank]
#             print('global round:{:d}, client id={:d}, localE={:d}'.format(rnd, rank, local_epochs))
            
#             local_model = copy.deepcopy(global_model)
#             criterion = nn.CrossEntropyLoss().to(device)
#             optimizer = selected_opt(local_model.parameters(),
#                              lr=args.lr,
#                              gmf=args.gmf,
#                              mu=args.mu,
#                              ratio=DataRatios[rank],
#                              momentum=args.momentum,
#                              nesterov = False,
#                              weight_decay=1e-4)
            
#             update_learning_rate(optimizer, rnd, args.lr)
            
            
#             for t in range(local_epochs):
#                 w, gn=train(local_model, criterion, optimizer, train_loader, t)
#                 local_weights.append(copy.deepcopy(w))
#                 gradient_norm.append(copy.deepcopy(gn))
                
        
#         global_weights = average_weights(local_weights)
#         global_model.load_state_dict(global_weights)
        
#         test_acc, test_loss = evaluate(global_model, 
#                                        test_loader)
#         print('global round:{:d}, test_acc={:.2f}, test_loss={:.4f}'.format(rnd, test_acc, test_loss))
#         test_accs.append(test_acc)
#         test_losses.append(test_loss)
        
#         # save results
#         csvfile = open('./log/'+args.exp_name+'/epoch_acc_loss.csv',"a",newline = "")      #w是覆盖形写入，a是追加写入
#         writer = csv.writer(csvfile)
#         writer.writerow([rnd, test_acc, test_loss])
#         csvfile.close()
        




    